In [ ]:
# !pip install gpytorch transformers

import torch
import torch.nn as nn
import gpytorch
from gpytorch.models import ApproximateGP
from gpytorch.variational import CholeskyVariationalDistribution, VariationalStrategy
from gpytorch.kernels import Kernel, MaternKernel, RBFKernel
from transformers import Adafactor
import numpy as np
import time
from sklearn.metrics import confusion_matrix, cohen_kappa_score, accuracy_score


# ==========================================
# 1. Custom Composite Covariance Function
# ==========================================
class GaussianMaternKernel(Kernel):
    """
    Composite kernel combining Matern (h=5/2) and RBF (Gaussian)
    as described in the G-MDRF paper.
    """

    has_lengthscale = True

    def __init__(self, **kwargs):
        super(GaussianMaternKernel, self).__init__(**kwargs)
        # The paper specifies h = 5/2 for the Matern component
        self.matern = MaternKernel(nu=2.5, **kwargs)
        self.rbf = RBFKernel(**kwargs)

        # Fluctuation parameter xi (ξ) from the paper
        self.register_parameter(
            name="raw_xi", parameter=torch.nn.Parameter(torch.tensor(0.0))
        )

    @property
    def xi(self):
        return torch.nn.functional.softplus(self.raw_xi)

    def forward(self, x1, x2, diag=False, **params):
        # k(x, x') = Matern(x, x') * (xi^2 * RBF(x, x'))
        matern_covar = self.matern(x1, x2, diag=diag, **params)
        rbf_covar = self.rbf(x1, x2, diag=diag, **params)

        if diag:
            return matern_covar * (self.xi**2) * rbf_covar
        else:
            return matern_covar.mul(rbf_covar) * (self.xi**2)


# ==========================================
# 2. Spectral Field Modeling (SIVGP)
# ==========================================
class SIVGPModel(ApproximateGP):
    def __init__(self, inducing_points, num_classes):
        # Construct the variational distribution and strategy
        variational_distribution = CholeskyVariationalDistribution(
            inducing_points.size(0), batch_shape=torch.Size([num_classes])
        )
        variational_strategy = (
            gpytorch.variational.IndependentMultitaskVariationalStrategy(
                VariationalStrategy(
                    self,
                    inducing_points,
                    variational_distribution,
                    learn_inducing_locations=True,
                ),
                num_tasks=num_classes,
            )
        )
        super(SIVGPModel, self).__init__(variational_strategy)

        self.mean_module = gpytorch.means.ConstantMean(
            batch_shape=torch.Size([num_classes])
        )
        self.covar_module = gpytorch.kernels.ScaleKernel(
            GaussianMaternKernel(batch_shape=torch.Size([num_classes])),
            batch_shape=torch.Size([num_classes]),
        )

    def forward(self, x):
        mean_x = self.mean_module(x)
        covar_x = self.covar_module(x)
        return gpytorch.distributions.MultivariateNormal(mean_x, covar_x)


# ==========================================
# 3. Spatial Field Modeling (SAMRF via ADMM)
# ==========================================
def samrf_admm_patch(f_gp, patch_features, delta_sp=10.0, phi=0.1, max_iter=10):
    """
    Localized ADMM optimization for the SAMRF spatial prior over a 9x9 patch.
    f_gp: Spectral latent function predictions [batch_size, num_classes]
    patch_features: The 9x9 local context [batch_size, 1, 50, 9, 9]
    """
    batch_size, num_classes = f_gp.shape
    device = f_gp.device

    # Initialize variables per Algorithm 1
    t = f_gp.clone()  # t^(0) = f_GP
    Q1 = t.clone()
    Q3 = t.clone()
    H1 = torch.zeros_like(t)
    H3 = torch.zeros_like(t)

    # Precompute a localized gradient magnitude for adaptive weighting
    # We use the center pixel vs its immediate patch neighborhood as a rough local gradient
    center_idx = patch_features.shape[-1] // 2
    center_pixels = patch_features[:, 0, :, center_idx, center_idx]
    patch_mean = patch_features.mean(dim=(3, 4))[:, 0, :]
    local_grad = torch.norm(center_pixels - patch_mean, p=1, dim=1, keepdim=True)
    adaptive_alpha = torch.exp(-local_grad)  # Simple edge-preserving weight

    for i in range(max_iter):
        # 1. Update t
        # Based on Eq 32: t^(i+1) <- (1 / (2*phi + 1)) * (f_GP + phi*(Q1 + H1 + Q3 + H3))
        t_new = (1.0 / (2 * phi + 1)) * (f_gp + phi * (Q1 + H1 + Q3 + H3))

        # 2. Update auxiliary variables (Q) using Soft Thresholding for L1 spatial constraints
        # Localized approximation of spatial gradient penalty
        threshold = (delta_sp * adaptive_alpha) / phi

        # Soft thresholding operator for Q1 (representing spatial smoothness)
        Q1_new = torch.sign(t_new - H1) * torch.clamp(
            torch.abs(t_new - H1) - threshold, min=0.0
        )

        # Non-negativity constraint for Q3 (iota_R+)
        Q3_new = torch.clamp(t_new - H3, min=0.0)

        # 3. Update multipliers (H)
        H1_new = H1 + t_new - Q1_new
        H3_new = H3 + t_new - Q3_new

        # Check convergence (simplified)
        if torch.norm(t_new - t) < 1e-4:
            t = t_new
            break

        t, Q1, Q3, H1, H3 = t_new, Q1_new, Q3_new, H1_new, H3_new

    return t


# ==========================================
# 4. G-MDRF Pipeline & Input Adaptation
# ==========================================
class GMDRF_Pipeline(nn.Module):
    def __init__(self, train_loader, num_classes, spectral_dim, device):
        super().__init__()
        self.num_classes = num_classes
        self.device = device

        # --- Inducing Points Extraction (0.8% of training data) ---
        print("Extracting inducing points (~0.8% of training samples)...")
        all_center_pixels = []
        for data, _ in train_loader:
            # Extract center pixel: [batch_size, 1, 50, 9, 9] -> [batch_size, 50]
            center = data[:, 0, :, 4, 4]
            all_center_pixels.append(center)

        full_train_x = torch.cat(all_center_pixels, dim=0)
        num_inducing = max(int(0.008 * full_train_x.size(0)), 10)  # At least 10 points
        indices = torch.randperm(full_train_x.size(0))[:num_inducing]
        inducing_points = full_train_x[indices].to(device)

        # Initialize SIVGP
        self.gp_model = SIVGPModel(inducing_points, num_classes).to(device)
        self.likelihood = gpytorch.likelihoods.MultitaskGaussianLikelihood(
            num_tasks=num_classes
        ).to(device)

    def train_step(self, x_patch, y, optimizer, mll):
        self.gp_model.train()
        self.likelihood.train()

        # Data Adaptation: Unwrap [B, 1, C, H, W] to center pixel [B, C]
        center_idx = x_patch.shape[-1] // 2
        x_center = x_patch[:, 0, :, center_idx, center_idx].float()

        optimizer.zero_grad()
        output = self.gp_model(x_center)

        # Convert classification labels to one-hot for GP regression approximation
        y_one_hot = torch.nn.functional.one_hot(y, num_classes=self.num_classes).float()

        loss = -mll(output, y_one_hot)
        loss.backward()
        optimizer.step()
        return loss.item()

    def evaluate(self, test_loader):
        self.gp_model.eval()
        self.likelihood.eval()

        all_preds = []
        all_targets = []

        print("Running SIVGP + SAMRF Evaluation...")
        with torch.no_grad(), gpytorch.settings.fast_pred_var():
            for data, target in test_loader:
                data, target = data.to(self.device).float(), target.to(self.device)

                # 1. Spectral Field (SIVGP)
                center_idx = data.shape[-1] // 2
                x_center = data[:, 0, :, center_idx, center_idx]
                predictions = self.likelihood(self.gp_model(x_center))
                f_gp = predictions.mean  # Latent function predictions

                # 2. Spatial Field (SAMRF via ADMM)
                # Applying the spatial constraint over the continuous function space
                t_spatial = samrf_admm_patch(f_gp, data, delta_sp=10.0, phi=0.1)

                # Softmax to get final class
                pred_class = torch.argmax(t_spatial, dim=1)

                all_preds.extend(pred_class.cpu().numpy())
                all_targets.extend(target.cpu().numpy())

        # Calculate Metrics
        all_preds = np.array(all_preds)
        all_targets = np.array(all_targets)

        oa = accuracy_score(all_targets, all_preds)
        kappa = cohen_kappa_score(all_targets, all_preds)
        cm = confusion_matrix(all_targets, all_preds)

        # Average Per-Class Accuracy (AA)
        per_class_acc = cm.diagonal() / cm.sum(axis=1)
        aa = np.nanmean(per_class_acc)

        return oa, aa, kappa, per_class_acc


# ==========================================
# Execution Loop Configuration
# ==========================================
def run_gmdrf(train_loader, test_loader, num_classes, spectral_dim, epochs=50):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    pipeline = GMDRF_Pipeline(train_loader, num_classes, spectral_dim, device)

    # Use Adafactor for the GP parameters as specified in the paper
    optimizer = Adafactor(
        [
            {"params": pipeline.gp_model.parameters()},
            {"params": pipeline.likelihood.parameters()},
        ],
        lr=1e-2,
        relative_step=False,
        scale_parameter=False,
        warmup_init=False,
    )

    # Marginal Log Likelihood for Variational GP
    mll = gpytorch.mlls.VariationalELBO(
        pipeline.likelihood, pipeline.gp_model, num_data=len(train_loader.dataset)
    )

    print(f"Starting Adafactor Optimization for SIVGP...")
    for epoch in range(epochs):
        epoch_loss = 0
        start_time = time.time()

        for data, target in train_loader:
            data, target = data.to(device), target.to(device)
            loss = pipeline.train_step(data, target, optimizer, mll)
            epoch_loss += loss

        print(
            f"Epoch {epoch + 1:03d}/{epochs} | Loss: {epoch_loss / len(train_loader):.4f} | Time: {time.time() - start_time:.2f}s"
        )

    # Final Evaluation combining Spectral and Spatial fields
    oa, aa, kappa, per_class_acc = pipeline.evaluate(test_loader)

    print("\n" + "=" * 40)
    print(f"G-MDRF FINAL RESULTS")
    print("=" * 40)
    print(f"Overall Accuracy (OA): {oa * 100:.2f}%")
    print(f"Average Accuracy (AA): {aa * 100:.2f}%")
    print(f"Cohen's Kappa (K):     {kappa:.4f}")
    print("=" * 40)


# Assuming you have initialized your dataloaders as `train_loader` and `test_loader`:
# run_gmdrf(train_loader, test_loader, num_classes=16, spectral_dim=50, epochs=50)
